In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "85bd815efd86af24338a0f0c45844cb7e1e6f155"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/85bd815efd86af24338a0f0c45844cb7e1e6f155/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Momentum

In that section, we saw why plain gradient descent struggles on an ill-conditioned valley: the steepest curvature $\lambda_{\max}$ imposes the stability ceiling $\eta < 2/\lambda_{\max}$, forcing the flattest direction $\lambda_{\min}$ to contract by at most $O(1/\kappa)$ per step, where $\kappa = \lambda_{\max}/\lambda_{\min}$ is the condition number. Plain gradient descent is slow because each step relies solely on the instantaneous gradient $\mathbf{g}_t$. Yet the history of recent gradients carries clear geometric information: along the valley floor, successive gradients point consistently in the same direction, whereas across the steep walls they alternate in sign as the iterate overshoots from side to side.

@Polyak.1964 proposed stepping along an exponentially weighted sum of past gradients—called the **velocity**—so that persistent components along flat directions accumulate while oscillating components across steep directions cancel. Analogous to a heavy ball rolling down a trough against friction, **heavy-ball momentum** reduces the worst-case iteration count on a quadratic bowl from $O(\kappa)$ to $O(\sqrt{\kappa})$ at the cost of storing a single velocity buffer per parameter.

In this section, we analyze the velocity recursion, its effective step size and memory timescale, and the $\sqrt{\kappa}$ acceleration law via critical damping. We then test momentum on full-batch softmax regression and minibatch training of `FashionCNN`, introduce Nesterov look-ahead momentum, and establish practical rules for tuning $\beta$ and $\eta$.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## The Momentum Method

### Velocity and Effective Step Size

Given gradient $\mathbf{g}_t$ at step $t$ (exact or minibatch), heavy-ball momentum maintains a velocity vector $\mathbf{v}_t$ initialized at $\mathbf{v}_0 = \mathbf{0}$ and updates:

$$
\begin{aligned}
\mathbf{v}_t &= \beta\, \mathbf{v}_{t-1} + \mathbf{g}_t, \\
\mathbf{x}_{t+1} &= \mathbf{x}_t - \eta\, \mathbf{v}_t,
\end{aligned}
$$

where $\beta \in [0, 1)$ is the momentum coefficient ($\beta = 0$ recovers gradient descent). Unrolling the recursion expresses the velocity as an exponentially weighted sum of all past gradients:

$$\mathbf{v}_t = \sum_{k=0}^{t-1} \beta^k \mathbf{g}_{t-k} = \mathbf{g}_t + \beta\, \mathbf{g}_{t-1} + \beta^2\, \mathbf{g}_{t-2} + \cdots + \beta^{t-1}\, \mathbf{g}_1.$$

Consider what this weighted sum does to the two coordinates of the valley $f(\mathbf{x}) = 0.1 x_1^2 + 2 x_2^2$ ($\lambda_1 = 0.2, \lambda_2 = 4$) at $\eta = 0.6$—above the plain gradient descent stability limit $2/\lambda_2 = 0.5$, where plain gradient descent diverges along $x_2$ by factor $1 - 4(0.6) = -1.4$ per step. Adding momentum $\beta = 0.5$ stabilizes the trajectory and brings both coordinates within $0.01$ of the origin in $20$ steps:

In [ ]:
h1, h2 = d2l.VALLEY_CURVATURES
f_valley, valley_grad = d2l.f_valley, d2l.valley_grad

def momentum_2d(x1, x2, v1, v2, f_grad, eta, beta):
    g1, g2 = f_grad(x1, x2)
    v1, v2 = beta * v1 + g1, beta * v2 + g2
    return x1 - eta * v1, x2 - eta * v2, v1, v2

d2l.show_trace_2d(f_valley, d2l.train_2d(momentum_2d, valley_grad, eta=0.6,
                                         beta=0.5))

Two limiting cases explain why momentum simultaneously accelerates the flat coordinate and stabilizes the steep one:

* **Persistent gradients (flat floor):** When successive gradients remain nearly constant ($\mathbf{g}_{t-k} \approx \mathbf{g}$), the velocity builds up to $\mathbf{v}_t \approx (1 + \beta + \beta^2 + \cdots)\mathbf{g} = \frac{1}{1-\beta}\mathbf{g}$, yielding the **effective step size**
  $$\eta_\textrm{eff} = \frac{\eta}{1-\beta}.$$
* **Alternating gradients (steep walls):** When successive gradients alternate in sign ($\mathbf{g}_{t-k} \approx (-1)^k \mathbf{g}$), the geometric series sums to $1 - \beta + \beta^2 - \cdots = \frac{1}{1+\beta}$, damped to an effective step size of $\frac{\eta}{1+\beta}$.

At $\eta = 0.6$ and $\beta = 0.5$, momentum amplifies progress along the valley floor to $\eta_{\text{eff}} = 0.6 / 0.5 = 1.2$ while damping oscillations across the steep wall to $0.6 / 1.5 = 0.4 < 0.5$. Consequently, the stability ceiling for curvature $\lambda$ rises from $\eta\lambda < 2$ to $\eta\lambda < 2(1 + \beta)$ (Exercise 3).

### The Timescale of $\beta$ and Minibatch Noise

Because the weights $1, \beta, \beta^2, \ldots$ sum to $1/(1-\beta)$, the velocity has an effective memory window of $\tau = 1/(1-\beta)$ steps and a half-life of $\log(1/2)/\log\beta$ steps:

In [ ]:
print('beta   window   half-life   weight inside the window')
for beta in [0.5, 0.9, 0.99]:
    window = 1 / (1 - beta)
    half_life = np.log(0.5) / np.log(beta)
    inside = 1 - beta ** window
    print(f'{beta:4}   {window:6.0f}   {half_life:9.1f}   {inside:24.2f}')

Increasing $\beta$ from $0.9$ to $0.99$ expands both the memory horizon and the effective step size $\eta_{\text{eff}} = \eta/(1-\beta)$ tenfold (from $10$ steps to $100$ steps).

Under stochastic gradients $\mathbf{g}_t = \nabla f(\mathbf{x}_t) + \boldsymbol{\xi}_t$, does averaging over $1/(1-\beta)$ steps shrink the stationary noise ball of that section? No: each noise perturbation $\boldsymbol{\xi}_t$ enters $\mathbf{v}_t$ with weight $1$ and decays as $\beta^k$ over subsequent steps, contributing a cumulative displacement of $\eta(1 + \beta + \beta^2 + \cdots)\boldsymbol{\xi}_t = \eta_{\text{eff}}\boldsymbol{\xi}_t$. Thus, at matched effective step size $\eta_{\text{eff}} = \eta/(1-\beta)$, SGD with and without momentum settle into stationary noise balls of nearly identical variance (Exercise 5). When increasing $\beta$ in a noise-limited run, $\eta$ should therefore be scaled proportional to $1 - \beta$ to hold $\eta_{\text{eff}}$ constant.

## Acceleration and Critical Damping

What choice of $(\eta, \beta)$ is optimal on a quadratic with eigenvalues $\lambda_{\min} \le \cdots \le \lambda_{\max}$ and condition number $\kappa = \lambda_{\max}/\lambda_{\min}$? @Polyak.1964 proved that setting

$$
\eta^\star = \frac{4}{\left(\sqrt{\lambda_{\max}} + \sqrt{\lambda_{\min}}\right)^2},
\qquad
\beta^\star = \left(\frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1}\right)^{2}
$$

achieves the worst-case per-step contraction factor

$$\rho^\star = \sqrt{\beta^\star} = \frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1} \approx 1 - \frac{2}{\sqrt{\kappa}},$$

reducing the steps needed to reach relative accuracy $\epsilon$ from $\frac{\kappa}{2}\log(1/\epsilon)$ to $\frac{\sqrt{\kappa}}{2}\log(1/\epsilon)$ (that section).

To see why the contraction factor equals $\sqrt{\beta^\star}$, substitute $\eta v_t = x_t - x_{t+1}$ into the 1D velocity update along an eigenvector of curvature $\lambda$ to obtain the second-order linear recurrence

$$x_{t+1} = (1 + \beta - \eta\lambda)\, x_t - \beta\, x_{t-1},$$

whose characteristic polynomial is $z^2 - (1 + \beta - \eta\lambda)z + \beta = 0$. By Vieta's formulas, the product of the two roots is $z_1 z_2 = \beta$. Whenever the discriminant $\Delta = (1 + \beta - \eta\lambda)^2 - 4\beta$ is non-positive—which occurs for all $\beta \ge \beta_{\text{c}}(\lambda) = (1 - \sqrt{\eta\lambda})^2$—the roots form a complex-conjugate pair of identical modulus $|z_1| = |z_2| = \sqrt{\beta}$, independent of the curvature $\lambda$ (the figure).

![Per-step contraction factor of heavy-ball momentum along curvature $\lambda$ as a function of $\beta$. Convergence is fastest at the critical damping threshold $\beta_\textrm{c} = (1 - \sqrt{\eta\lambda})^2$; below $\beta_\textrm{c}$ the mode is overdamped, while above $\beta_\textrm{c}$ it oscillates with modulus $\sqrt{\beta}$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-opt-critical-damping.svg)

The optimal parameters the equation equate the critical damping thresholds of the flattest and steepest eigenvalues, $(1 - \sqrt{\eta^\star\lambda_{\min}})^2 = (\sqrt{\eta^\star\lambda_{\max}} - 1)^2 = \beta^\star$, critically damping both extreme modes while ensuring that every intermediate eigenvalue $\lambda \in [\lambda_{\min}, \lambda_{\max}]$ lies in the underdamped complex-root regime and contracts at rate $\sqrt{\beta^\star}$.

On our valley ($\kappa = 20$), the equation yields $\eta^\star \approx 0.668, \beta^\star \approx 0.403$, reducing the asymptotic contraction factor from $0.905$ to $\sqrt{\beta^\star} \approx 0.635$ and cutting the steps to reach $10^{-6}$ relative error from $139$ down to $40$:

In [ ]:
def steps_to(tol, eta, beta, steps=400):
    """Steps after which |x1|, |x2| stay below tol times their start."""
    x1, x2, v1, v2, last = -5.0, -2.0, 0.0, 0.0, 0
    for t in range(1, steps + 1):
        x1, x2, v1, v2 = momentum_2d(x1, x2, v1, v2, valley_grad, eta, beta)
        if abs(x1) > 5 * tol or abs(x2) > 2 * tol:
            last = t
    return last + 1

def contraction(eta, beta):
    return max(abs(np.roots([1, -(1 + beta - eta * lam), beta])).max()
               for lam in (h1, h2))

eta_opt = 4 / (np.sqrt(h2) + np.sqrt(h1)) ** 2
beta_opt = ((np.sqrt(h2 / h1) - 1) / (np.sqrt(h2 / h1) + 1)) ** 2
print('  eta   beta   factor   steps to 1%   steps to 1e-6')
for eta, beta in [(2 / (h1 + h2), 0), (0.6, 0.5), (eta_opt, beta_opt),
                  (0.3, 0.8)]:
    print(f'{eta:5.3f}  {beta:5.3f}  {contraction(eta, beta):7.3f}  '
          f'{steps_to(1e-2, eta, beta):12d}  {steps_to(1e-6, eta, beta):14d}')

Notice the fourth row ($\eta = 0.3, \beta = 0.8$): choosing $\beta$ well above the critical threshold $\beta_{\text{c}}$ leaves both coordinates underdamped with modulus $\sqrt{0.8} \approx 0.894$, causing the trajectory to overshoot and ring around the origin:

In [ ]:
d2l.show_trace_2d(f_valley, d2l.train_2d(momentum_2d, valley_grad, eta=0.3,
                                         beta=0.8))

## Momentum on Data

Our analysis predicts two distinct regimes when training on data:

1. **Curvature-limited regime (accurate gradients, large $\kappa$):** Momentum permits a larger effective step $\eta_{\text{eff}}$ and reduces the required iteration count by a large factor.
2. **Noise-limited regime (small minibatches):** The stationary noise floor is governed by $\eta_{\text{eff}} = \eta/(1-\beta)$, so adding $\beta = 0.9$ shifts the optimal learning rate down by a factor of $1 - \beta = 0.1$ while achieving comparable convergence speed.

We verify both regimes on Fashion-MNIST by comparing $\beta = 0$ and $\beta = 0.9$ on (1) full-batch softmax regression ($b = 60{,}000$, exact gradients, ill-conditioned due to non-negative pixel intensities) and (2) `FashionCNN` (that section) with small minibatches ($b = 32$):

In [ ]:
fashion = d2l.FashionMNIST()
X = jnp.asarray(fashion.train[0][..., None], jnp.float32) / 255  # NHWC
y = jnp.asarray(fashion.train[1], jnp.int32)

class Batches:
    """The examples X, y on the device, in a new random order on every pass."""
    def __init__(self, X, y, batch_size):
        self.X, self.y, self.batch_size = X, y, batch_size
        self.key = d2l.get_key()  # Advanced on every pass

    def train_dataloader(self):
        self.key, key = jax.random.split(self.key)
        order = jax.random.permutation(key, len(self.y))
        return ((self.X[i], self.y[i])
                for i in order.reshape(-1, self.batch_size))

In [ ]:
def softmax_regression(rngs):
    """A linear layer from the pixels to the logits, starting at zero."""
    return nnx.Sequential(lambda X: X.reshape(len(X), -1),
                          nnx.Linear(28 * 28, 10, rngs=rngs,
                                     kernel_init=nnx.initializers.zeros))

make_sgd = lambda beta: lambda lr: optax.sgd(lr, momentum=beta)
new_seed = d2l.get_seed

In [ ]:
# Model, batch size, steps, target loss
setups = {'softmax regression': (softmax_regression, 60000, 1000, 0.5),
          'FashionCNN': (d2l.FashionCNN, 32, 3000, 0.3)}
grids = {('softmax regression', 0.0): [0.0625, 0.125, 0.25, 0.5, 1],
         ('softmax regression', 0.9): [0.1, 0.2, 0.4, 0.8],
         ('FashionCNN', 0.0): [0.05, 0.1, 0.2, 0.4],
         ('FashionCNN', 0.9): [0.005, 0.01, 0.02, 0.04]}
best = {}
for (name, beta), lrs in grids.items():
    make_model, b, num_steps, target = setups[name]
    curves, _ = d2l.lr_sweep(make_model, make_sgd(beta), Batches(X, y, b),
                             lrs, new_seed(), num_steps, verbose=False)
    best[name, beta] = min(lrs, key=lambda lr: d2l.final_loss(curves[lr]))
    c = curves[best[name, beta]]
    print(f'{name:<18}  beta={beta}  best lr {best[name, beta]:<5g}  steps to '
          f'{target}: {d2l.steps_to_reach(c, target)}  final loss '
          f'{d2l.final_loss(c):.3f}')

Both predictions hold quantitatively:

* On full-batch **softmax regression**, adding momentum $\beta = 0.9$ increases the optimal learning rate from $\eta = 0.125$ to $\eta = 0.4$ ($\eta_{\text{eff}} = 4.0$, a $32\times$ larger effective step), cutting the steps needed to reach loss $0.5$ by more than $6\times$ ($625 \to 100$ steps) and lowering the final loss from $0.472$ to $0.365$.
* On small-batch **`FashionCNN`** ($b = 32$), the run is noise-limited: the optimal learning rate drops from $\eta \approx 0.1$–$0.2$ at $\beta = 0$ to $\eta \approx 0.02$–$0.04$ at $\beta = 0.9$—a factor of $5$–$10$ that keeps the effective step $\eta_{\text{eff}}$ within one step of the factor-of-two grid of its $\beta = 0$ value—and both configurations reach loss $0.3$ and final losses of $0.26$–$0.30$ within the single-seed variation of that section.

## Nesterov Momentum

In the underdamped run ($\eta = 0.3, \beta = 0.8$), heavy-ball momentum overshot the minimum because it evaluated the gradient at the current position $\mathbf{x}_t$ rather than at the point where the velocity was about to carry it. @Nesterov.1983 proposed evaluating the gradient at the **look-ahead point** $\mathbf{x}_t - \eta\beta\mathbf{v}_{t-1}$ (the figure):

$$
\begin{aligned}
\mathbf{v}_t &= \beta\, \mathbf{v}_{t-1} + \nabla f(\mathbf{x}_t - \eta \beta\, \mathbf{v}_{t-1}), \\
\mathbf{x}_{t+1} &= \mathbf{x}_t - \eta\, \mathbf{v}_t.
\end{aligned}
$$

![One step of heavy-ball versus Nesterov momentum from the same iterate and velocity. Heavy-ball evaluates the gradient at $\mathbf{x}_t$ and overshoots; Nesterov evaluates the gradient at the look-ahead point $\mathbf{x}_t - \eta\beta\mathbf{v}_{t-1}$, applying an immediate restoring correction.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-opt-nesterov-lookahead.svg)

Evaluating the gradient at the look-ahead point introduces anticipatory damping that suppresses steep-direction oscillations and achieves the optimal $O(1/t^2)$ convergence rate on general smooth convex objectives [@Nesterov.2018] (that section):

In [ ]:
def nesterov_2d(x1, x2, v1, v2, f_grad, eta, beta):
    g1, g2 = f_grad(x1 - eta * beta * v1,  # Gradient at the look-ahead
                    x2 - eta * beta * v2)  # point, not at (x1, x2)
    v1, v2 = beta * v1 + g1, beta * v2 + g2
    return x1 - eta * v1, x2 - eta * v2, v1, v2

d2l.show_trace_2d(f_valley, d2l.train_2d(nesterov_2d, valley_grad, eta=0.3,
                                         beta=0.8))

In practice, $\beta = 0.9$ is a robust default across most architectures, with the learning rate tuned via the effective step $\eta_{\text{eff}} = \eta/(1-\beta)$. However, momentum still applies a single scalar step size to all coordinates. When gradient scales differ by orders of magnitude across layers—as in Transformers—we need per-coordinate adaptive scaling (that section).

## Summary

* Heavy-ball momentum steps along an exponentially decaying velocity $\mathbf{v}_t = \beta\mathbf{v}_{t-1} + \mathbf{g}_t$ with memory horizon $1/(1-\beta)$. It amplifies persistent gradient directions to an effective step $\eta_{\text{eff}} = \eta/(1-\beta)$ while damping alternating directions to $\eta/(1+\beta)$, raising the stability ceiling to $\eta\lambda < 2(1+\beta)$.
* On a quadratic with condition number $\kappa$, tuning $(\eta^\star, \beta^\star)$ critically damps the extreme eigenvalues and reduces the iteration count from $O(\kappa)$ to $O(\sqrt{\kappa})$.
* Nesterov momentum evaluates the gradient at the look-ahead point $\mathbf{x}_t - \eta\beta\mathbf{v}_{t-1}$, damping underdamped oscillations.
* Under minibatch noise, momentum preserves the stationary noise ball at matched effective step $\eta_{\text{eff}} = \eta/(1-\beta)$, delivering large speedups when curvature dominates and matching plain SGD at $\eta \leftarrow (1-\beta)\eta$ when minibatch noise dominates.

## Exercises

1. [code] **Momentum and learning rate grid.** Run `momentum_2d` on `f_valley` across $\beta \in \{0, 0.25, 0.5, 0.8\}$ and $\eta \in \{0.1, 0.3, 0.6, 1.0\}$ for up to $1{,}000$ steps. Tabulate the steps required to reach $\|\mathbf{x}_t\| \le 10^{-3}$ (or divergence) and compare the empirical optimum at each $\eta$ against $\beta^\star$ the equation and $\beta_{\text{c}}(\lambda) = (1 - \sqrt{\eta\lambda})^2$.

1. **Velocity noise-to-signal ratio.** For $\mathbf{g}_t = \mathbf{g} + \boldsymbol{\xi}_t$ with i.i.d. zero-mean noise of variance $\sigma^2$ per coordinate, show that $\mathbf{v}_t$ in the equation has stationary mean $\mathbf{g}/(1-\beta)$ and stationary variance $\sigma^2/(1-\beta^2)$, reducing the velocity's noise-to-signal ratio by $\sqrt{(1-\beta)/(1+\beta)}$.

1. **Equivalent forms of Nesterov momentum and stability.**
    1. Show by induction that the library form $\mathbf{v}_t = \beta\mathbf{v}_{t-1} + \nabla f(\mathbf{y}_t)$, $\mathbf{y}_{t+1} = \mathbf{y}_t - \eta(\nabla f(\mathbf{y}_t) + \beta\mathbf{v}_t)$ produces the exact same velocities as the equation with $\mathbf{y}_t = \mathbf{x}_t - \eta\beta\mathbf{v}_{t-1}$.
    1. Prove from the equation that both roots of $z^2 - (1 + \beta - \eta\lambda)z + \beta = 0$ lie strictly inside the unit circle for $0 \le \beta < 1$ if and only if $0 < \eta\lambda < 2(1+\beta)$.

1. **The exponentially averaged momentum form.** Compare the equation against the normalized EMA form $\mathbf{m}_t = \beta\mathbf{m}_{t-1} + (1-\beta)\mathbf{g}_t$, $\mathbf{x}_{t+1} = \mathbf{x}_t - \alpha\mathbf{m}_t$ used in Adam (that section). Show that $\mathbf{m}_t = (1-\beta)\mathbf{v}_t$, so both trajectories coincide when $\alpha = \eta/(1-\beta) = \eta_{\text{eff}}$.

1. [code] **Momentum under minibatch noise across batch sizes.** ●
    1. On $f(x) = \frac{\lambda}{2}x^2$ with noisy gradient $g_t = \lambda x_t + \xi_t$, compare the stationary iterate variance of plain SGD at step size $\alpha$ against momentum ($\beta = 0.9$) at $\eta = (1-\beta)\alpha$ for $\alpha\lambda \in \{0.05, 0.2\}$.
    1. Train `d2l.FashionCNN` for $1{,}000$ steps at batch sizes $b \in \{32, 500, 6000\}$ with $\beta \in \{0, 0.9\}$ at matched effective step $\eta/(1-\beta) = 0.2$, then sweep $\eta$ at $b = 6000$ to verify that momentum accelerates training once the batch size is large enough to enter the curvature-limited regime.

[Discussions](https://d2l.discourse.group/t/1071)